<a href="https://colab.research.google.com/github/ardominguezm/golden-age-semantic-reconfiguration/blob/main/notebooks/paper1_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 1 — Golden Age Semantic Reconfiguration

**Current stage: Phase 11 — Semantic Network Construction & Structural Comparability**

## Scientific target retained
The paper asks whether the Renaissance→Baroque transition is merely gradual semantic drift or whether it involves **structural reorganization of the relations among poetic concepts**. The empirical object is therefore a dynamic **concept↔concept semantic network**, not an author-similarity classifier.

## Novelty guardrail
The contribution is operationalized through four linked elements: (i) composition time reconstructed independently of the semantic analysis and propagated as chronological uncertainty; (ii) concept-level relations rather than author/text similarity; (iii) an explicit later decomposition of **lexical turnover** versus **relational rewiring among persistent concepts**; and (iv) raw versus **author-balanced** networks so that Góngora's corpus share cannot by itself masquerade as historical change. Historiographic labels and 1580/1605 remain external validation only.

Phase 11 builds the first semantic networks, but inspects them **only for structural feasibility and comparability**. It does not estimate a change point, rank historiographic markers, or interpret a temporal peak as Renaissance/Baroque evidence.


In [ ]:
# Reproducible environment
import sys, subprocess, hashlib, urllib.request, re, shutil, unicodedata, math
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from functools import lru_cache
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

SPACY_VERSION='3.8.7'
MODEL_NAME='es_core_news_sm'
MODEL_VERSION='3.8.0'
NETWORKX_VERSION='3.4.2'
MODEL_URL='https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl'
MODEL_SHA256='e451a83d6df79b87e9eed0cb553f03e99e36a3bab18a7b79f0dcfd1fdf875e12'
wheel=Path('/content/es_core_news_sm-3.8.0-py3-none-any.whl')
if not wheel.exists() or hashlib.sha256(wheel.read_bytes()).hexdigest()!=MODEL_SHA256:
    urllib.request.urlretrieve(MODEL_URL,wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest()==MODEL_SHA256
subprocess.run([sys.executable,'-m','pip','install','-q',f'spacy=={SPACY_VERSION}',f'networkx=={NETWORKX_VERSION}',str(wheel)],check=True)
import spacy, networkx as nx
assert spacy.__version__==SPACY_VERSION,spacy.__version__
assert nx.__version__==NETWORKX_VERSION,nx.__version__
nlp=spacy.load(MODEL_NAME,disable=['parser','ner'])
assert nlp.meta.get('version')==MODEL_VERSION,nlp.meta
print('Environment ready:',f'spaCy {spacy.__version__}',f'| {MODEL_NAME} {MODEL_VERSION}',f'| networkx {nx.__version__}')


In [ ]:
# Reproduce the exact Phase-10 source reconstruction and frozen Phase-8 primary chronology.
SOURCES={
 'navarro_tei':('https://github.com/bncolorado/CorpusSonetosSigloDeOro.git','092a5fe70a4065a4d84bfed288bffd3851348f9c'),
 'gongora_scholarly':('https://github.com/gongoradigital/gongoraobra.git','3beadeecc059a7cc48499dc2683bb378a2630978'),
}
ROOT=Path('/content/gasr_phase11_sources'); ROOT.mkdir(exist_ok=True)
def clone(name,url,commit):
    dst=ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    subprocess.run(['git','clone','--quiet',url,str(dst)],check=True)
    subprocess.run(['git','-C',str(dst),'checkout','--quiet',commit],check=True)
    got=subprocess.check_output(['git','-C',str(dst),'rev-parse','HEAD'],text=True).strip()
    assert got==commit,(name,got,commit)
    return dst
paths={k:clone(k,*v) for k,v in SOURCES.items()}
N=paths['navarro_tei']; G=paths['gongora_scholarly']
XML_ID='{http://www.w3.org/XML/1998/namespace}id'
def local(tag): return tag.split('}')[-1] if '}' in tag else tag
def el_text(el): return '' if el is None else ' '.join(' '.join(el.itertext()).split())
def norm(s):
    s=unicodedata.normalize('NFKD',str(s)); s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]','',s.lower())
def years_1580_1626(s):
    return sorted(set(int(x) for x in re.findall(r'(?<!\d)(1[56]\d{2})(?!\d)',str(s)) if 1580<=int(x)<=1626))
rows=[]
for fp in sorted(N.rglob('*.xml')):
    root=ET.parse(fp).getroot(); lines=[el_text(x) for x in root.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    author=fp.parent.name; txt='\n'.join(lines)
    rows.append({'n_id':f'{author}::{fp.name}','author_dir':author,'source_file':str(fp.relative_to(N)),'n_lines':len(lines),'lines':lines,'text_tei':txt,'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),'first_line':lines[0]})
n=pd.DataFrame(rows); assert len(n)==5078,len(n)

primary_rows=[]
def add(pid,author,lo,hi,confidence,basis):
    primary_rows.append({'n_id':pid,'author_dir':author,'composition_min':int(lo),'composition_max':int(hi),'temporal_confidence':confidence,'temporal_basis':basis})

# Góngora: exact pinned scholarly linkage used in Phases 4/5/10.
groot=ET.parse(G/'gongora_obra-poetica.xml').getroot(); parent={child:par for par in groot.iter() for child in par}; grows=[]
for el in groot.iter():
    xid=el.attrib.get(XML_ID,'')
    if local(el.tag)!='div' or not xid.lower().startswith('poem'): continue
    lines=[el_text(x) for x in el.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    vals=[]; cur=el
    for _ in range(6):
        vals+=list(cur.attrib.values())
        if cur.text: vals.append(cur.text)
        for ch in list(cur):
            if local(ch.tag) in {'head','date','label'}: vals.append(el_text(ch))
            if ch.tail: vals.append(ch.tail)
        cur=parent.get(cur)
        if cur is None: break
    ys=sorted(set(y for v in vals for y in years_1580_1626(v))); txt='\n'.join(lines)
    grows.append({'g_id':xid,'n_lines':len(lines),'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),'scholarly_year':ys[0] if len(ys)==1 else pd.NA,'year_status':'unique' if len(ys)==1 else ('ambiguous' if len(ys)>1 else 'missing')})
g=pd.DataFrame(grows); g14=g[(g.n_lines==14)&g.signature.ne('')].copy(); g_by_id=g.set_index('g_id',drop=False); ng=n[n.author_dir.eq('Gongora')].copy(); sig_to_gids=g14.groupby('signature').g_id.apply(list).to_dict(); links=[]
for r in ng.itertuples(index=False):
    exact_ids=sig_to_gids.get(r.signature,[])
    if len(exact_ids)==1: gid,score,method=exact_ids[0],1.0,'exact'
    else:
        best_gid,best_score=None,-1.0
        for gr in g14.itertuples(index=False):
            sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
            if sc>best_score: best_gid,best_score=gr.g_id,sc
        gid,score,method=best_gid,best_score,'fuzzy'
    links.append({'n_id':r.n_id,'g_id':gid,'method':method,'score':float(score),'preaccept':method=='exact' or score>=0.98})
glink=pd.DataFrame(links); pre=glink[glink.preaccept].copy(); collisions=set(pre.g_id.value_counts()[lambda s:s>1].index); glink['accept_phase4']=glink.preaccept&~glink.g_id.isin(collisions)
acc=glink[glink.accept_phase4].merge(g[['g_id','scholarly_year','year_status']],on='g_id',how='left'); acc=acc[acc.year_status.eq('unique')&acc.scholarly_year.notna()].copy()
for r in acc.itertuples(index=False): add(r.n_id,'Gongora',r.scholarly_year,r.scholarly_year,'A' if r.method=='exact' else 'B','scholarly_chronology_year_exact_link' if r.method=='exact' else 'scholarly_chronology_year_fuzzy_link')
phase4_nids=set(acc.n_id); phase4_gids=set(acc.g_id); unmatched=ng[~ng.n_id.isin(phase4_nids)].copy(); first2_index=g14.groupby('first2_signature').g_id.apply(list).to_dict(); recovered=[]
for r in unmatched.itertuples(index=False):
    ids2=first2_index.get(r.first2_signature,[])
    if len(ids2)!=1: continue
    gid=ids2[0]
    if gid in phase4_gids: continue
    gr=g_by_id.loc[gid]; score=SequenceMatcher(None,r.signature,gr.signature).ratio()
    if score>=0.95 and gr.year_status=='unique' and pd.notna(gr.scholarly_year): recovered.append((r.n_id,gid,score,int(gr.scholarly_year)))
rec=pd.DataFrame(recovered,columns=['n_id','g_id','score','year']); dup=set(rec.g_id.value_counts()[lambda s:s>1].index) if len(rec) else set(); rec=rec[~rec.g_id.isin(dup)]
for r in rec.itertuples(index=False): add(r.n_id,'Gongora',r.year,r.year,'B','scholarly_chronology_year_variant_link')
assert sum(x['author_dir']=='Gongora' for x in primary_rows)==58

# Garcilaso: exact frozen Phase-10 intervals/anchors.
GAR={**{i:(1526,1532,'B','scholarly_phase_interval') for i in [1,2,3,4,6,26,27]},25:(1534,1535,'B','scholarly_interval'),33:(1535,1535,'A','historically_anchored_scholarly_year'),35:(1535,1535,'A','historically_anchored_scholarly_year'),**{i:(1533,1535,'B','revised_scholarly_interval') for i in [7,8,12,15,19,28,30,31]}}
for no,(lo,hi,conf,basis) in GAR.items(): add(f'GarcilasoDeLaVega::GarcilasoDeLaVega_{no:02d}.xml','GarcilasoDeLaVega',lo,hi,conf,basis)
for no,lo,hi,conf,basis in [(30,1596,1596,'B','Cadiz_1596'),(13,1598,1598,'A','FelipeII_tomb_1598'),(31,1597,1598,'B','Herrera_death_epitaph')]: add(f'Cervantes::Cervantes_{no}.xml','Cervantes',lo,hi,conf,basis)
for no,lo,hi,basis in [(224,1574,1574,'Alameda_CarlosV'),(279,1578,1579,'Barahona_Granada'),(276,1573,1574,'Bazan_Tunis'),(300,1580,1582,'Portugal_to_H'),(281,1578,1578,'DonJuan_de_Austria')]: add(f'FernandoDeHerrera::FernandoDeHerrera_{no}.xml','FernandoDeHerrera',lo,hi,'B',basis)
for no in [2,19,4,5]: add(f'PedroEspinosa::PedroEspinosa_{no}.xml','PedroEspinosa',1594,1596,'B','Espinosa_happiness_period_1594_1596')
for no,year,basis in [(131,1609,'Carrillo_sonnet_1609'),(69,1611,'Aminta_1611'),(70,1611,'Aminta_1611'),(72,1611,'Aminta_1611'),(76,1611,'Aminta_1611'),(42,1610,'HenryIV_1610'),(43,1610,'HenryIV_1610'),(45,1610,'HenryIV_1610'),(44,1624,'Osuna_1624')]: add(f'Quevedo::Quevedo_{no}.xml','Quevedo',year,year,'B',basis)
primary=pd.DataFrame(primary_rows).drop_duplicates('n_id').copy(); expected={'Gongora':58,'GarcilasoDeLaVega':18,'Quevedo':9,'FernandoDeHerrera':5,'PedroEspinosa':4,'Cervantes':3}
assert len(primary)==97 and primary.groupby('author_dir').size().to_dict()==expected
primary=primary.merge(n[['n_id','text_tei','lines','n_lines','source_file','first_line','signature']],on='n_id',how='left',validate='one_to_one'); assert primary.text_tei.notna().all()
print('PHASE-8 PRIMARY CHRONOLOGY REPRODUCED:',len(primary),'poems |',primary.author_dir.nunique(),'authors')
display(primary.groupby('author_dir').size().rename('primary_poems').to_frame())


In [ ]:
# Reproduce Phase-10 preprocessing exactly; freeze global concept vocabulary before networks.
MAIN_POS={'NOUN','VERB','ADJ','ADV'}; MIN_LEMMA_LEN=2
line_records=[]
for r in primary.itertuples(index=False):
    for line_no,line in enumerate(r.lines,1): line_records.append((r.n_id,r.author_dir,line_no,line))
docs=list(nlp.pipe([x[3] for x in line_records],batch_size=128)); assert len(docs)==len(line_records); token_rows=[]
for (pid,author,line_no,_),doc in zip(line_records,docs):
    for t in doc:
        lemma=unicodedata.normalize('NFC',str(t.lemma_)).strip().lower(); pos=t.pos_; keep=bool(t.is_alpha and pos in MAIN_POS and len(lemma)>=MIN_LEMMA_LEN)
        token_rows.append({'n_id':pid,'author_dir':author,'line_no':line_no,'surface':t.text,'lemma':lemma,'pos':pos,'is_alpha':bool(t.is_alpha),'is_main_content':keep,'concept':f'{lemma}::{pos}' if keep else pd.NA})
tokens=pd.DataFrame(token_rows); main_tok=tokens[tokens.is_main_content].copy()
concept_df=main_tok[['n_id','concept','lemma','pos']].drop_duplicates(['n_id','concept']).groupby(['concept','lemma','pos']).n_id.nunique().rename('poem_df').reset_index(); concept_tf=main_tok.groupby('concept').size().rename('token_frequency').reset_index(); vocab=concept_df.merge(concept_tf,on='concept',how='left')
MAIN_VOCAB=set(vocab.loc[vocab.poem_df.ge(2),'concept']); SENS_VOCAB=set(vocab.loc[vocab.poem_df.ge(3),'concept'])
assert len(MAIN_VOCAB)==668,len(MAIN_VOCAB); assert len(SENS_VOCAB)==349,len(SENS_VOCAB); assert len(tokens)==10439 and int(tokens.is_main_content.sum())==4358,(len(tokens),int(tokens.is_main_content.sum()))
poem_line_sets={r.n_id:[set() for _ in range(r.n_lines)] for r in primary.itertuples(index=False)}
for (pid,line_no),grp in main_tok[main_tok.concept.isin(MAIN_VOCAB)].groupby(['n_id','line_no']): poem_line_sets[pid][int(line_no)-1]=set(grp.concept)
print('Phase-10 representation reproduced exactly')
print('Tokens:',len(tokens),'| retained content:',int(tokens.is_main_content.sum()),'| main vocabulary df>=2:',len(MAIN_VOCAB),'| sensitivity df>=3:',len(SENS_VOCAB))


## Main network definition

For a temporal window $W$, each retained concept is a node. Within each poetic line, concept presence is binary: repeated occurrences of the same concept in the same line do not multiply its contribution.

For raw networks, $c_i$ is the number of poetic lines containing $i$, $c_{ij}$ is the number containing both $i,j$, and $N_W$ is the total number of poetic lines. Pairs require $c_{ij}\ge2$ and edge weight is $\mathrm{PPMI}(i,j)=\max[0,\log_2\{p(i,j)/(p(i)p(j))\}]$.

For author-balanced networks, a line from poem $p$ by author $a$ receives $w_{\ell,p,a,W}=1/(n_{a,W}L_p)$, so every author present contributes total line mass 1. The raw two-line support requirement is retained before weighted PPMI is evaluated.

Active nodes are retained even when isolated after PPMI thresholding. Phase 11 treats graph size/connectivity as QA, not as historical evidence.


In [ ]:
# Network construction functions
primary_idx=primary.set_index('n_id',drop=False); MIN_PAIR_SUPPORT=2
def build_network(ids_tuple,mode='raw',return_detail=False):
    ids=list(ids_tuple)
    if not ids: raise ValueError('Empty temporal window')
    sub=primary_idx.loc[ids]; author_counts=Counter(sub.author_dir); n_poems=len(ids); n_authors=len(author_counts); shares=np.array(list(author_counts.values()),dtype=float)/n_poems; effective_authors=float(1/np.square(shares).sum()); top_author_share=float(shares.max())
    node_mass=defaultdict(float); pair_mass=defaultdict(float); raw_support=defaultdict(int); total_mass=0.0; total_lines=0
    for pid in ids:
        row=primary_idx.loc[pid]; lines=poem_line_sets[pid]; L=len(lines); author=row.author_dir; line_weight=1.0 if mode=='raw' else 1.0/(author_counts[author]*L); total_lines+=L; total_mass+=L*line_weight
        for concepts in lines:
            concepts=sorted(concepts)
            for u in concepts: node_mass[u]+=line_weight
            for u,v in combinations(concepts,2): pair_mass[(u,v)]+=line_weight; raw_support[(u,v)]+=1
    if mode=='author_balanced': assert np.isclose(total_mass,n_authors),(total_mass,n_authors)
    active=sorted(u for u,c in node_mass.items() if c>0); G=nx.Graph(); G.add_nodes_from(active); edge_rows=[]
    for (u,v),support in raw_support.items():
        if support<MIN_PAIR_SUPPORT: continue
        pij=pair_mass[(u,v)]/total_mass; pi=node_mass[u]/total_mass; pj=node_mass[v]/total_mass
        if pij<=0 or pi<=0 or pj<=0: continue
        ppmi=max(0.0,math.log2(pij/(pi*pj)))
        if ppmi>0:
            G.add_edge(u,v,weight=ppmi,raw_support=support,weighted_support=pair_mass[(u,v)])
            if return_detail: edge_rows.append({'u':u,'v':v,'ppmi':ppmi,'raw_support':support,'weighted_support':pair_mass[(u,v)]})
    nn=G.number_of_nodes(); ne=G.number_of_edges(); components=list(nx.connected_components(G)) if nn else []; gcc=max((len(c) for c in components),default=0); weights=[d['weight'] for _,_,d in G.edges(data=True)]; supports=[d['raw_support'] for _,_,d in G.edges(data=True)]
    metrics={'n_poems':n_poems,'n_authors':n_authors,'effective_authors':effective_authors,'top_author_share':top_author_share,'n_lines':total_lines,'line_mass':total_mass,'n_nodes':nn,'vocab_coverage':nn/len(MAIN_VOCAB),'n_edges':ne,'density':nx.density(G) if nn>1 else np.nan,'n_components':len(components),'gcc_fraction':gcc/nn if nn else np.nan,'mean_degree':(2*ne/nn) if nn else np.nan,'median_ppmi':float(np.median(weights)) if weights else np.nan,'median_raw_support':float(np.median(supports)) if supports else np.nan}
    if not return_detail: return metrics
    node_rows=[{'concept':u,'degree':G.degree(u),'strength':sum(d['weight'] for _,_,d in G.edges(u,data=True))} for u in G.nodes()]
    return metrics,pd.DataFrame(edge_rows),pd.DataFrame(node_rows)
@lru_cache(maxsize=None)
def cached_metrics(ids_tuple,mode): return build_network(ids_tuple,mode,False)
print('Network constructors ready | main raw pair support >=',MIN_PAIR_SUPPORT,'poetic lines')


In [ ]:
# Propagate Phase-9 chronological uncertainty through all 9 main windows.
SEED=20260825; MC_DRAWS=1000; MAIN_WINDOWS=[(s,s+19) for s in range(1565,1606,5)]; assert len(MAIN_WINDOWS)==9
ids=primary.n_id.to_numpy(object); lo=primary.composition_min.to_numpy(int); hi=primary.composition_max.to_numpy(int); rng=np.random.default_rng(SEED); sampled=np.empty((MC_DRAWS,len(primary)),dtype=int)
for j,(a,b) in enumerate(zip(lo,hi)): sampled[:,j]=a if a==b else rng.integers(a,b+1,size=MC_DRAWS)
records=[]
for m in range(MC_DRAWS):
    yrs=sampled[m]
    for start,end in MAIN_WINDOWS:
        mask=(yrs>=start)&(yrs<=end); selected=tuple(sorted(ids[mask].tolist()))
        for mode in ('raw','author_balanced'):
            recm={'draw':m,'start':start,'end':end,'mode':mode}; recm.update(cached_metrics(selected,mode)); records.append(recm)
    if (m+1)%100==0: print('completed',m+1,'/',MC_DRAWS,'chronology draws')
mc=pd.DataFrame(records); metric_cols=['n_poems','n_authors','effective_authors','top_author_share','n_lines','n_nodes','vocab_coverage','n_edges','density','n_components','gcc_fraction','mean_degree','median_ppmi','median_raw_support']; summary_rows=[]
for (start,end,mode),grp in mc.groupby(['start','end','mode'],sort=True):
    row={'start':start,'end':end,'mode':mode}
    for col in metric_cols:
        x=grp[col].astype(float); row[col+'_median']=float(x.median()); row[col+'_q10']=float(x.quantile(.10)); row[col+'_q90']=float(x.quantile(.90))
    summary_rows.append(row)
structural_summary=pd.DataFrame(summary_rows)
print('\nPHASE 11 STRUCTURAL NETWORK AUDIT'); print('(medians across chronology draws; QA only, not historical inference)')
display(structural_summary[['start','end','mode','n_poems_median','n_authors_median','effective_authors_median','top_author_share_median','n_nodes_median','n_edges_median','density_median','gcc_fraction_median']])
print('Cache:',cached_metrics.cache_info())


In [ ]:
# Midpoint-year reference graphs: reproducibility/visual QA only, NOT inferential estimates.
mid=((lo+hi)//2); edge_exports=[]; node_exports=[]; ref_metrics=[]
for start,end in MAIN_WINDOWS:
    selected=tuple(sorted(ids[(mid>=start)&(mid<=end)].tolist()))
    for mode in ('raw','author_balanced'):
        metrics,edges,nodes=build_network(selected,mode,True); metrics.update({'start':start,'end':end,'mode':mode}); ref_metrics.append(metrics)
        if not edges.empty: edge_exports.append(edges.assign(start=start,end=end,mode=mode))
        if not nodes.empty: node_exports.append(nodes.assign(start=start,end=end,mode=mode))
reference_metrics=pd.DataFrame(ref_metrics); reference_edges=pd.concat(edge_exports,ignore_index=True) if edge_exports else pd.DataFrame(); reference_nodes=pd.concat(node_exports,ignore_index=True) if node_exports else pd.DataFrame()
overlap=[]
for start,end in MAIN_WINDOWS:
    er=reference_edges[(reference_edges.start==start)&(reference_edges.end==end)&reference_edges['mode'].eq('raw')]; eb=reference_edges[(reference_edges.start==start)&(reference_edges.end==end)&reference_edges['mode'].eq('author_balanced')]; R=set(map(tuple,er[['u','v']].to_numpy())); B=set(map(tuple,eb[['u','v']].to_numpy())); union=R|B
    overlap.append({'start':start,'end':end,'raw_edges':len(R),'balanced_edges':len(B),'shared_edges':len(R&B),'edge_jaccard_raw_vs_balanced':len(R&B)/len(union) if union else np.nan})
weighting_overlap=pd.DataFrame(overlap)
OUT=Path('/content/gasr_phase11_outputs'); OUT.mkdir(exist_ok=True)
mc.to_csv(OUT/'phase11_mc_network_metrics.csv',index=False); structural_summary.to_csv(OUT/'phase11_mc_structural_summary.csv',index=False); reference_metrics.to_csv(OUT/'phase11_reference_network_metrics.csv',index=False); reference_edges.to_csv(OUT/'phase11_reference_edges.csv',index=False); reference_nodes.to_csv(OUT/'phase11_reference_nodes.csv',index=False); weighting_overlap.to_csv(OUT/'phase11_raw_vs_author_balanced_overlap.csv',index=False)
assert len(structural_summary)==18; assert set(structural_summary['mode'])=={'raw','author_balanced'}; assert structural_summary.n_nodes_median.gt(0).all(); assert structural_summary.n_edges_median.gt(0).all(); assert np.isfinite(structural_summary.gcc_fraction_median).all()
print('\nRAW vs AUTHOR-BALANCED reference overlap (QA only)'); display(weighting_overlap)
print('\nPHASE 11 CHECKPOINT'); print('-------------------')
print('Primary chronology:',len(primary),'poems |',primary.author_dir.nunique(),'authors'); print('Main trajectory:',len(MAIN_WINDOWS),'windows | 20 years | step 5'); print('Chronology realizations:',MC_DRAWS); print('Main vocabulary:',len(MAIN_VOCAB),'concepts | line support >=',MIN_PAIR_SUPPORT); print('Network modes: raw + author_balanced'); print('All 18 window×mode structural summaries non-degenerate: TRUE'); print('Temporal graph distance / rewiring statistic computed: FALSE'); print('Change point computed: FALSE'); print('1580/1605 used to tune networks: FALSE'); print('Historiographic labels used to tune networks: FALSE'); print('Next scientific target: decompose lexical turnover vs relational rewiring under chronology uncertainty'); print('Outputs:',OUT)
